## Dataset inmuebles publicados de Mercado Libre

In [ ]:
import pandas as pd

In [ ]:
df_original =  pd.read_csv("Base_ML.csv")

df_original

# EDA: Análisis de Datos Exploratorio 

In [ ]:
df_original.info()

In [ ]:
df_original.shape

In [ ]:
df_original.info

In [ ]:
df_original.head()

## Cuenta: sirve para hacer conteos dentro de la tabla**

In [ ]:
df_original.tail(5)

In [ ]:
df_original.describe()

In [ ]:
df_original.describe(include = "all")

In [ ]:
df =  df_original.copy()

In [ ]:
## Conversion de fecha

df["Fecha"] = pd.to_datetime(df["Fecha"])

df.info

In [ ]:
df["Fecha"].min()

In [ ]:
df["Fecha"].max()

## Exploración de datos null

In [ ]:
#df.isna().sum().sort_values(normalize = True)

In [ ]:
df.isna().mean().sort_values(ascending = False) * 100

## Imputación de valores

In [ ]:
filtro =   df["Antiguedad"].isna()
df[filtro]

## ¿Sería necesario imputar la columna antigüedad?
## Validar si hay correlacioón con otras columnas


# Renombre de columnas

In [ ]:
df = df.rename(columns={
    "Número de Publicación": "numero_publicacion",
    "Fecha": "fecha",
    "Tipo de Inmueble": "tipo_inmueble",
    "Uso": "uso",
    "Título": "titulo",
    "Operación": "operacion",
    "Condición": "condicion",
    "Precio": "precio_original",
    "Precio en USD": "precio_usd",
    "Divisa": "divisa",
    "Precio en MXN": "precio_mxn",
    "Superficie Total": "superficie_total",
    "Superficie Construida": "superficie_construida",
    "Unidad Superficie": "unidad_superficie",
    "Superficie total en m²": "superficie_m2",
    "Antiguedad": "antiguedad",
    "Ubicación": "ubicacion",
    "Municipio": "municipio",
    "Estado": "estado",
    "año-mes": "anio_mes",
    "trimestre": "trimestre",
    "Cuenta": "cuenta"
})

df.head()

## Analisis de datos: Univariado

- Identificar los diferentes tipos de inmuebles

In [ ]:
df["tipo_inmueble"].value_counts()

- Los diferentes tipos de uso

In [ ]:
df["uso"].value_counts()

In [ ]:
df["operacion"].value_counts()

In [ ]:
df["condicion"].value_counts()

In [ ]:
df["precio_original"].mean()

## Creación de nuevas columnas

- `anio`: año de publicación.
- `mes`: mes de publicación.
- `precio_por_m2`: precio en MXN dividido entre superficie en m².

In [ ]:
df["anio"] = df["fecha"].dt.year
df["mes"] = df["fecha"].dt.month

df["precio_por_m2"] = df["precio_mxn"] / df["superficie_total"]
df

# Filtros para análisis de datos

In [ ]:
filtro =  df["operacion"] == "Renta"

df[filtro]

In [ ]:
filtro =  df["operacion"] == "Venta"

df[filtro]

- Seleccionar los departamentos rentados en Ciudad de México

In [ ]:
filtro =  (df["operacion"] == "Renta") & (df["estado"]=="Ciudad de México")
df[filtro].sort_values(by="precio_original", ascending = False)

In [ ]:
filtro =  (df["operacion"] == "Venta") & (df["estado"]=="Ciudad de México")
df[filtro].sort_values(by="precio_original", ascending = False)

- Obtener las casas y departamentos ubicaos en ciudad de México

In [ ]:
filtro =  (df["tipo_inmueble"].isin(["Casa","Departamento"]) ) &  (df["estado"] == "Ciudad de México")

df[filtro].sort_values(by="precio_original", ascending =  False)

- Ordenar lo anterior por precio original y hacer el conteo de cada tipo de inmueble

In [ ]:
df[filtro].sort_values(by="precio_original", ascending =  False)["tipo_inmueble"].value_counts()

- Publicaciones que inlcuyan la palabra jardín

In [ ]:
filtro = df["titulo"].str.contains("jardín",
                                   case = False,
                                   na = False)

df[filtro]["titulo"]

- Extraer el número de recámaras desde el título

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
df_viviendas = df[(df["uso"] == "Vivienda")]
#df_viviendas
#df_viviendas["num_recamaras"] = df_viviendas["titulo"].str.lower().str.extract(r"([\d+])\s*rec")
#df_viviendas["num_recamaras"] = df_viviendas["titulo"].str.lower().str.extract(r"(\s[a-z|0-9]+)\s*[rec]")
df_viviendas["num_recamaras"] = (
    df_viviendas["titulo"]
    .str.lower()
    .str.extract(r"(\d+)\s*(?:rec|recámaras?|recamaras?|habitaciones?|hab\.?)")
)

df_viviendas[["titulo","num_recamaras"]].head()

- Precio en pesos mexicanos promedio por el tipo de inmueble

In [ ]:
df.groupby("tipo_inmueble")["precio_mxn"].mean()

In [ ]:
df.columns

- Conteo de numero de publicacione, precio en pesos mexicanos promedio, la media del precio, precio minimo, precio máximo, y superficie promedio por estado , todo junto

In [ ]:
resumen = (df.groupby("estado")
           .agg(
               publicaciones =  ('numero_publicacion', 'count'),
               promedio_preciomxn = pd.NamedAgg (column = "precio_mxn", aggfunc = "mean"),
               media_precio_mxn = ('precio_mxn', 'median'),
               precio_min = ('precio_mxn', 'min'),
               precio_max  = ('precio_mxn', 'max'),
               superficie_promedo = ('superficie_m2', 'mean'),
               
           ))

resumen

- Precio mínimo, promedio y máiximo por estado. Todo junot

# Uniones y merge

In [ ]:
tabla_regiones = pd.DataFrame({
    "estado": [
        "Ciudad de México", "México", "Jalisco", "Nuevo León", "Yucatán",
        "Quintana Roo", "Baja California", "Puebla", "Querétaro", "Guanajuato"
    ],
    "region": [
        "Centro", "Centro", "Occidente", "Norte", "Sureste",
        "Sureste", "Norte", "Centro", "Bajío", "Bajío"
    ]
})

tabla_regiones

In [ ]:
merge_left =  df.merge( ## df izquierdo = df , df derecho =  tabla_Regiones
    tabla_regiones,
    how = "left",
    on = "estado"
)
merge_left

In [ ]:
merge_right=  df.merge( ## df izquierdo = df , df derecho =  tabla_Regiones
    tabla_regiones,
    how = "right",
    on = "estado"
)
merge_right

In [ ]:
merge_inner =  df.merge( ## df izquierdo = df , df derecho =  tabla_Regiones
    tabla_regiones,
    how = "inner",
    on = "estado"
)
merge_inner

In [ ]:
merge_outer =  df.merge( ## df izquierdo = df , df derecho =  tabla_Regiones
    tabla_regiones,
    how = "outer",
    on = "estado"
)
merge_outer

# Join

In [ ]:
df_index =  df.set_index("estado")
df_index



In [ ]:
tabla_reg_index = tabla_regiones.set_index("estado")

tabla_reg_index

In [ ]:
res_join =  df_index.join(tabla_reg_index,
                         how = "left" )

res_join

# Concat

In [ ]:
datos_renta = df[df["operacion"] == "Renta"]
datos_venta = df[df["operacion"] == "Venta"]

In [ ]:
concatenacion =  pd.concat([datos_renta, datos_venta], axis = 1)

concatenacion

In [ ]:
concatenacion =  pd.concat([datos_renta, datos_venta], axis = 0)

concatenacion